# Google Colab H100 実行セットアップ

## 📋 事前準備

1. Google Colab を開く: https://colab.research.google.com/
2. ランタイム → ランタイムのタイプを変更
3. ハードウェアアクセラレータ: **GPU**
4. GPU タイプ: **A100** または **H100** (Colab Pro+)
5. 保存

## ⏱️ 実行時間とコスト

- **A100 (Colab Pro)**: 約18-22時間, $49.99/月
- **H100 (Colab Pro+)**: 約11-13時間, 利用可能な場合

## 1. Google Drive マウント

In [ ]:
from google.colab import drive
import os

# Google Drive をマウント
drive.mount('/content/drive')

# プロジェクトディレクトリのパスを設定
# ※ 自分のGoogle Driveのパスに変更してください
PROJECT_DIR = '/content/drive/MyDrive/MATWM_Project'

print(f'Project directory: {PROJECT_DIR}')

## 2. プロジェクトファイルのアップロード

### 方法A: Google Drive経由（推奨）

1. ローカルPCで以下のファイルをZIP圧縮:
   - `train_gamma_true.py`
   - `train_gamma_false.py`
   - `matwm_implementation.py`
   - `matwm_agent.py`
   - `matwm_utils.py`
   - `curiosity_reward.py`
   - `requirements.txt`

2. Google Drive の `MyDrive/` に `MATWM_Project.zip` をアップロード

3. 以下のセルを実行して解凍

In [ ]:
import zipfile

# ZIPファイルのパス
zip_path = '/content/drive/MyDrive/MATWM_Project.zip'

# 解凍先ディレクトリ
extract_to = '/content/matwm_project'

# 解凍
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f'✓ Extracted to: {extract_to}')
    
    # ファイル一覧を表示
    !ls -lh {extract_to}
else:
    print(f'✗ ZIP file not found: {zip_path}')
    print('Please upload MATWM_Project.zip to Google Drive')

### 方法B: 直接アップロード

In [ ]:
# 左サイドバーのファイルアイコンから直接アップロード
# または以下のコードでアップロード

from google.colab import files

# 作業ディレクトリ作成
!mkdir -p /content/matwm_project
%cd /content/matwm_project

# ファイルアップロード（複数回実行して全ファイルをアップロード）
uploaded = files.upload()

print('Uploaded files:')
!ls -lh

## 3. GPU確認

In [ ]:
import torch

print('=' * 70)
print('GPU Information')
print('=' * 70)
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('⚠️  GPU not available. Please change runtime type.')

# nvidia-smi
!nvidia-smi

## 4. 依存パッケージのインストール

In [ ]:
# 作業ディレクトリに移動
%cd /content/matwm_project

# requirements.txt からインストール
!pip install -q -r requirements.txt

# インストール確認
!pip list | grep -E "torch|pettingzoo|gymnasium"

print('\n✓ Package installation complete')

## 5. 動作確認

In [ ]:
# インポートテスト
import sys
sys.path.insert(0, '/content/matwm_project')

from matwm_implementation import MATWMConfig
from matwm_agent import MATWMAgent
from curiosity_reward import CuriosityConfig

print('✓ All imports successful')

# 簡単な設定テスト
config = MATWMConfig(
    total_steps=100,
    warmup_steps=10,
)

print(f'✓ Config created: total_steps={config.total_steps}')

## 6. トレーニング実行（use_gamma_progress=True）

### ⚠️ 重要な注意事項

- Colabは**最大12時間**で切断されます
- 定期的にチェックポイントをGoogle Driveに保存します
- 切断された場合は、チェックポイントから再開できます

In [ ]:
# 作業ディレクトリに移動
%cd /content/matwm_project

# train_gamma_true.py を実行
!python train_gamma_true.py 2>&1 | tee train_true.log

## 7. 結果の保存（Google Driveに自動バックアップ）

In [ ]:
import shutil
from datetime import datetime

# バックアップ先ディレクトリ
backup_dir = '/content/drive/MyDrive/MATWM_Results'
os.makedirs(backup_dir, exist_ok=True)

# タイムスタンプ
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 結果をコピー
if os.path.exists('/content/matwm_project/results_gamma_true'):
    dest = os.path.join(backup_dir, f'results_gamma_true_{timestamp}')
    shutil.copytree('/content/matwm_project/results_gamma_true', dest)
    print(f'✓ Results backed up to: {dest}')
else:
    print('✗ Results directory not found')

# ログファイルもコピー
if os.path.exists('/content/matwm_project/train_true.log'):
    shutil.copy('/content/matwm_project/train_true.log', 
                os.path.join(backup_dir, f'train_true_{timestamp}.log'))
    print(f'✓ Log file backed up')

## 8. チェックポイントから再開（切断された場合）

In [ ]:
# 最新のチェックポイントを探す
import glob

checkpoint_pattern = '/content/matwm_project/results_gamma_true/*/checkpoint_*/full_checkpoint.pt'
checkpoints = sorted(glob.glob(checkpoint_pattern))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f'Latest checkpoint: {latest_checkpoint}')
    
    # チェックポイントから再開するコードを追加
    # (train_gamma_true.py に resume_from パラメータを追加する必要があります)
else:
    print('No checkpoints found')

## 9. トレーニング実行（use_gamma_progress=False）

In [ ]:
# 作業ディレクトリに移動
%cd /content/matwm_project

# train_gamma_false.py を実行
!python train_gamma_false.py 2>&1 | tee train_false.log

## 10. 結果のダウンロード

In [ ]:
# 結果をZIP圧縮してダウンロード
!cd /content/matwm_project && zip -r results_gamma_true.zip results_gamma_true/
!cd /content/matwm_project && zip -r results_gamma_false.zip results_gamma_false/

# ダウンロード
from google.colab import files

files.download('/content/matwm_project/results_gamma_true.zip')
files.download('/content/matwm_project/results_gamma_false.zip')
files.download('/content/matwm_project/train_true.log')
files.download('/content/matwm_project/train_false.log')

## 💡 Tips

### Colab切断対策

1. **定期的なバックアップ**
   - 上記のバックアップセルを定期的に実行
   - チェックポイントは自動保存される（5000 steps毎）

2. **ブラウザを開いたままにする**
   - Colabは非アクティブで切断される可能性
   - ブラウザタブを開いたまま

3. **Colab Pro/Pro+を使用**
   - より長い実行時間
   - 優先的なGPUアクセス

### コスト

- **Colab Pro**: $9.99/月（A100アクセス）
- **Colab Pro+**: $49.99/月（H100アクセス、より長い実行時間）

### 実行時間

- **A100**: 約18-22時間（2セッション必要）
- **H100**: 約11-13時間（1セッション + α）